# NEU training launcher (Kaggle GPU)

Rendered from `notebooks/kaggle_train_launcher.ipynb` by `make kaggle-push` (ADR-019, ADR-020). Do not edit on Kaggle.

No training logic lives here: it clones the repo at a pinned commit, installs the lockfile and runs `src.training.train`.
Retraining (ADR-034) also attaches the private feedback dataset; the MD5 comes as a training argument.
No credentials either: training only writes files to `/kaggle/working/`. The laptop downloads them and publishes the run to MLflow (`make register`).

In [ ]:
COMMIT = "{{COMMIT}}"
REPO_URL = "{{REPO_URL}}"
UV_VERSION = "{{UV_VERSION}}"
NUM_WORKERS = "{{NUM_WORKERS}}"
TRAIN_ARGS = "{{TRAIN_ARGS}}".split()  # noqa: SIM905 (filled at render) e.g. --model NAME
REPO = "/tmp/repo"  # outside /kaggle/working, so the clone and .venv are not saved as output
OUT = "/kaggle/working"

In [ ]:
import subprocess
from pathlib import Path


def run(*cmd, cwd=None):
    subprocess.run(list(cmd), check=True, cwd=cwd)  # stop the notebook on any failure


run("pip", "install", "--quiet", f"uv=={UV_VERSION}")
run("git", "clone", "--quiet", REPO_URL, REPO)
run("git", "checkout", "--quiet", COMMIT, cwd=REPO)
run("git", "log", "-1", "--oneline", cwd=REPO)

In [ ]:
run("uv", "sync", "--locked", "--group", "train", cwd=REPO)

In [ ]:
raw_dir = next(Path("/kaggle/input").rglob("NEU-DET")).parent  # the private dataset
print("raw dir:", raw_dir)
feedback_args = []
if "--feedback-md5" in TRAIN_ARGS:  # retraining (ADR-034): the private feedback dataset
    found = sorted(Path("/kaggle/input").rglob("feedback.csv"))
    assert len(found) == 1, f"expected one feedback.csv, found {found}"
    feedback_csv = found[0]
    feedback_args = ["--feedback", str(feedback_csv)]
    print("feedback:", feedback_csv)
run(
    "uv",
    "run",
    "--locked",
    "--group",
    "train",
    "python",
    "-m",
    "src.training.train",
    "--config",
    "configs/params.yaml",
    "--raw-dir",
    str(raw_dir),
    "--out-dir",
    f"{OUT}/results/train",
    "--out-checkpoint",
    f"{OUT}/models/baseline/best.pt",
    "--num-workers",
    NUM_WORKERS,
    *TRAIN_ARGS,
    *feedback_args,
    cwd=REPO,
)